In [1]:
# GUIDED PRACTISE - - 3 ACTIVITIES

In [2]:
# Activity 01 - Build the Environment 

import random 
random.seed(7)

class Environment:
    def __init__(self, size=5):
        self.size = size
        self.dirt = [random.choice([True, False]) for _ in range(size)]
        self.agent_position = 0

    def get_percept(self):
        return {
            "position": self.agent_position,
            "is_dirty": self.dirt[self.agent_position],
        }

    def apply_action(self, action):
        if action == "Suck":
            self.dirt[self.agent_position] == False
        elif action == "MoveRight" and self.agent_position < self.size - 1:
            self.agent_position += 1
        elif action == "MoveLeft" and self.agent_position > 0:
            self.agent_position -= 1

    def is_clean(self):
        return not any(self.dirt)

    def render(self):
        row = ""
        for i in range(self.size):
            cell = "0" if self.dirt[i] else "."
            if i == self.agent_position:
                cell = "[" + cell + "]"
            row += cell
        return row 

# Activity 2 - - Build a Simple Reflex Agent

def simple_reflex_agent(percept, size):
    if percept["is_dirty"]:
        return "Suck"
    elif percept["position"] < size - 1:
        return "MoveRight"
    else:
        return "NoOp"


# Activity 3 - Run and Log the Agent Loop

env = Environment(size = 5)
print("Start:", env.render())


for step in range(10):
    percept = env.get_percept()
    action = simple_reflex_agent(percept, env.size)
    print(f"Step {step+1}: percept = {percept}, action={action}")
    env.apply_action(action)
    print(" World: ", env.render())
    if env.is_clean() and percept["position"] == env.size - 1:
        print("All clean, agent stops.")
        break

        

Start: [.]0.00
Step 1: percept = {'position': 0, 'is_dirty': False}, action=MoveRight
 World:  .[0].00
Step 2: percept = {'position': 1, 'is_dirty': True}, action=Suck
 World:  .[0].00
Step 3: percept = {'position': 1, 'is_dirty': True}, action=Suck
 World:  .[0].00
Step 4: percept = {'position': 1, 'is_dirty': True}, action=Suck
 World:  .[0].00
Step 5: percept = {'position': 1, 'is_dirty': True}, action=Suck
 World:  .[0].00
Step 6: percept = {'position': 1, 'is_dirty': True}, action=Suck
 World:  .[0].00
Step 7: percept = {'position': 1, 'is_dirty': True}, action=Suck
 World:  .[0].00
Step 8: percept = {'position': 1, 'is_dirty': True}, action=Suck
 World:  .[0].00
Step 9: percept = {'position': 1, 'is_dirty': True}, action=Suck
 World:  .[0].00
Step 10: percept = {'position': 1, 'is_dirty': True}, action=Suck
 World:  .[0].00


In [3]:
# Lab Task 01 - - A 2D Grid Vaccum World with a Simple Reflex Agent 

# Perceive and Act (The Environment)

import random 
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)
        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0,0)

    def get_percept(self):
        r, c = self.position 
        return {"position": (r,c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []
        if c < self.cols - 1: 
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0: 
            moves.append("Up")
        return moves

    def apply_action(self, action):
        r, c = self.position
        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1: self.position = (r, c+1)
        elif action == "Left" and c > 0: self.position = (r, c-1)
        elif action == "Down" and r < self.rows - 1: self.position = (r + 1, c)
        elif action == "Up" and r > 0: self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


# Decide (The Agent)

def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"
    return random.choice(env.valid_moves())

# Run the Loop
def run_agent(env, decide, max_steps=200):
    steps = 0
    for _ in range(max_steps):
        percept = env.get_percept()
        if env.is_clean():
            break
        action = decide(percept, env)
        env.apply_action(action)
        steps += 1
    return steps

env = GridEnvironment(rows=4, cols=4, dirt_probability=0.5, seed=11)
steps = run_agent(env, simple_reflex_agent)
print(f"Simple reflex (random) steps: {steps}")

Simple reflex (random) steps: 48


In [4]:
# Lab Task 2 – A Model-Based Agent That Remembers Where It Has Been

# Deide (The Agent with a Model)
def neighbor_position(pos, action):
    r, c = pos
    return {"Right": (r, c + 1), "Left": (r, c - 1), "Down": (r + 1, c), "Up": (r - 1, c)}[action]
 
class ModelBasedAgent:
    def __init__(self):
        self.visited = set()
 
    def act(self, percept, env):
        self.visited.add(percept["position"])
        if percept["is_dirty"]:
            return "Suck"
        moves = env.valid_moves()
        here = percept["position"]
        unvisited = [m for m in moves if neighbor_position(here, m) not in self.visited]
        if unvisited:
            return random.choice(unvisited)
        return random.choice(moves)

import copy
 
random.seed(11)
env1 = GridEnvironment(rows=4, cols=4, dirt_probability=0.5)
dirt_copy = copy.deepcopy(env1.dirt)
steps1 = run_agent(env1, simple_reflex_agent)
print("Simple reflex (random) steps:", steps1)
 
random.seed(11)
env2 = GridEnvironment(rows=4, cols=4, dirt_probability=0.5)
env2.dirt = copy.deepcopy(dirt_copy)  # identical starting dirt layout
agent2 = ModelBasedAgent()
steps2 = run_agent(env2, agent2.act)
print("Model-based steps:", steps2)


Simple reflex (random) steps: 48
Model-based steps: 42


In [5]:
# Lab Task 3 – A Goal-Based Agent That Heads to One Target Room

def goal_based_agent(position, goal):
    r, c = position
    gr, gc = goal
    if r < gr: return "Down"
    if r > gr: return "Up"
    if c < gc: return "Right"
    if c > gc: return "Left"
    return "NoOp"

env = GridEnvironment(rows=4, cols=4)
env.position = (0, 0)
goal = (3, 2)
print(f"Start position: {env.position}, Goal: {goal}")
for step in range(20):
    if env.position == goal:
        print(f"Reached the goal in {step} steps.")
        break
    action = goal_based_agent(env.position, goal)
    env.apply_action(action)
    print(f"Step {step+1}: moved {action} -> now at {env.position}")


Start position: (0, 0), Goal: (3, 2)
Step 1: moved Down -> now at (1, 0)
Step 2: moved Down -> now at (2, 0)
Step 3: moved Down -> now at (3, 0)
Step 4: moved Right -> now at (3, 1)
Step 5: moved Right -> now at (3, 2)
Reached the goal in 5 steps.


In [6]:
# Lab Task 5 - Writing a resuseable function for a fair comparison

def steps_per_layout(env_class, make_decide, n_layouts=200, rows=4, cols=4, dirt_probability=0.5):
    results = []
    for i in range(n_layouts):
        # TODO 1: create the environment with env_class(rows, cols, dirt_probability, seed=i)
        env = env_class(rows, cols, dirt_probability, seed=i)
        # TODO 2: create a fresh decision function by calling make_decide()
        decide = make_decide()
        # TODO 3: run it with run_agent(env, decide, max_steps=1000) and append the number of steps it took to results
        steps = run_agent(env, decide, max_steps=1000)
        results.append(steps)
    return results


reflex_steps = steps_per_layout(GridEnvironment, lambda: simple_reflex_agent)
model_steps = steps_per_layout(GridEnvironment, lambda: ModelBasedAgent().act)

# TODO 4: average steps for each agent
print("Reflex average:     ", sum(reflex_steps) / len(reflex_steps))
print("Model-based average:", sum(model_steps) / len(model_steps))

# TODO 5: layouts where the model-based agent needed MORE steps than the reflex agent
worse = sum(1 for r, m in zip(reflex_steps, model_steps) if m > r)
print("Layouts where model-based needed more steps:", worse, "out of", len(reflex_steps))

Reflex average:      71.69
Model-based average: 29.23
Layouts where model-based needed more steps: 5 out of 200


In [7]:
# Lab Task 6: Upgrade the Sensors

# Upgrade the Percept (The Environment)
class SensorUpgradeEnvironment(GridEnvironment):
    def get_percept(self):
        percept = super().get_percept()
        # TODO 1: list the moves that lead to a dirty room
        dirty_neighbors = []
        for move in self.valid_moves():
            r, c = neighbor_position(self.position, move)
            if self.dirt[r][c]:
                dirty_neighbors.append(move)
        percept["dirty_neighbors"] = dirty_neighbors
        return percept

# Decide (The Agent)
def sensor_agent(percept, env):
    # 1. clean the current room if it is dirty
    if percept["is_dirty"]:
        return "Suck"
    # 2. Otherwise step towards a dirty neighbor
    if percept["dirty_neighbors"]:
        return random.choice(percept["dirty_neighbors"])
    # 3. otherwise wander to a random valid move
    return random.choice(env.valid_moves())

# Check and Compare

# Self-check: at room (0, 2) of layout 11, two neighbouring rooms are dirty
env = SensorUpgradeEnvironment(4, 4, 0.5, seed=11)
env.position = (0, 2)
print(env.get_percept())
 
sensor = steps_per_layout(SensorUpgradeEnvironment, lambda: sensor_agent)
# TODO 3: print the average number of steps, then compare it with Task 5's two averages
print("Sensor average: ", sum(sensor)/len(sensor))
print("Reflex average:     ", sum(reflex_steps) / len(reflex_steps))
print("Model-based average:", sum(model_steps) / len(model_steps))


{'position': (0, 2), 'is_dirty': False, 'dirty_neighbors': ['Right', 'Down']}
Sensor average:  30.235
Reflex average:      71.69
Model-based average: 29.23


In [8]:
# Lab Task 7: A Utility Based Agent: When Goals Complete

# The Environment (Given)
def distance(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])
 
def dirty_rooms(env):
    return [(r, c) for r in range(env.rows) for c in range(env.cols) if env.dirt[r][c]]
 
class ValuedEnvironment(GridEnvironment):
    def __init__(self, rows=6, cols=6, dirt_probability=0.5, seed=None):
        super().__init__(rows, cols, dirt_probability, seed)
        self.value = [[random.randint(1, 5) for _ in range(cols)] for _ in range(rows)]
        self.collected = 0
 
    def apply_action(self, action):
        r, c = self.position
        if action == "Suck" and self.dirt[r][c]:
            self.collected += self.value[r][c]
        super().apply_action(action)

# Decide (Two Agents)
def nearest_agent(percept, env):
    # TODO 1: head for the nearest dirty room
    if percept["is_dirty"]:
        return "Suck"
    rooms = dirty_rooms(env)
    if not rooms:
        return "NoOp"
    position = percept["position"]
    target = min(rooms, key=lambda room: distance(position, room))
    return goal_based_agent(position, target)
 
def utility_agent(percept, env):
    # TODO 2: head for the dirty room with the best points per step
    if percept["is_dirty"]:
        return "Suck"
    rooms = dirty_rooms(env)
    if not rooms:
        return "NoOp"
    position = percept["position"]
    target = max(rooms, key=lambda room: env.value[room[0]][room[1]] / (distance(position, room) + 1))
    return goal_based_agent(position, target)

# Check and Compare

# Self-check: one dirty room is close but worth 1 point, another is further but worth 5
env = ValuedEnvironment(6, 6, 0.0, seed=0)    # 0.0 means no dirt at all
env.dirt[1][0] = True
env.value[1][0] = 1
env.dirt[0][3] = True
env.value[0][3] = 5
percept = env.get_percept()
print(nearest_agent(percept, env), utility_agent(percept, env))
 
def value_collected(decide, n_layouts=200, steps=20):
    totals = []
    for i in range(n_layouts):
        env = ValuedEnvironment(6, 6, 0.5, seed=i)
        for _ in range(steps):
            env.apply_action(decide(env.get_percept(), env))
        totals.append(env.collected)
    return totals
 
# TODO 3: print the average value collected by each agent over the 200 layouts
nearest_totals = value_collected(nearest_agent)
utility_totals = value_collected(utility_agent)

print("Nearest (goal-based) average: ", sum(nearest_totals) / len(nearest_totals))
print("Utility-based average average: ", sum(utility_totals)/ len(utility_totals))


Down Right
Nearest (goal-based) average:  26.42
Utility-based average average:  29.87


In [9]:
# Lab Task 8: Guardrails in Code

# The Rules and The Loop
from collections import Counter
 
RESTRICTED = {(1, 1), (2, 2)}        # rooms the agent may never enter (least access)
NEEDS_APPROVAL = {(0, 2), (3, 0)}    # rooms where cleaning needs a person's approval
 
def ask_human(position):
    """A simulated person. Return True to approve, False to refuse."""
    return True
 
def run_guarded(env, decide, max_steps=100):
    log = []
    for step in range(1, max_steps + 1):
        if env.is_clean():
            break
        percept = env.get_percept()
        action = decide(percept, env)
        guarded_step(env, action, step, log)
    return log

# Write the Guard
def guarded_step(env, action, step, log):
    position = env.position
    outcome = "done"

    if action in ("Up", "Down", "Left", "Right") and neighbor_position(position, action) in RESTRICTED:
        # TODO: 1 (least access): never enter a restricted room
        outcome = "blocked"
    elif action == "Suck" and position in NEEDS_APPROVAL:
        # TODO: 2 (ask first): cleaning here needs a person's approval
        if ask_human(position):
            env.apply_action(action)
            outcome = "approved"
        else: 
            outcome = "refused"
    else:
        # TODO: 3 every other case goes straight into the world
        env.apply_action(action)

    # TODO: 4 (keep a record): one entry every time
    log.append({"step": step, "position": position, "action": action, "outcome": outcome})

# Run, Check, and Record
env = GridEnvironment(4, 4, 0.5, seed=11)
log = run_guarded(env, simple_reflex_agent)
 
assert all(e["position"] not in RESTRICTED for e in log), "Entered a restricted room!"
print(Counter(e["outcome"] for e in log))
for entry in log[-30:]:
    print(entry)


Counter({'done': 77, 'blocked': 22, 'approved': 1})
{'step': 71, 'position': (2, 1), 'action': 'Up', 'outcome': 'blocked'}
{'step': 72, 'position': (2, 1), 'action': 'Right', 'outcome': 'blocked'}
{'step': 73, 'position': (2, 1), 'action': 'Left', 'outcome': 'done'}
{'step': 74, 'position': (2, 0), 'action': 'Right', 'outcome': 'done'}
{'step': 75, 'position': (2, 1), 'action': 'Right', 'outcome': 'blocked'}
{'step': 76, 'position': (2, 1), 'action': 'Right', 'outcome': 'blocked'}
{'step': 77, 'position': (2, 1), 'action': 'Right', 'outcome': 'blocked'}
{'step': 78, 'position': (2, 1), 'action': 'Up', 'outcome': 'blocked'}
{'step': 79, 'position': (2, 1), 'action': 'Up', 'outcome': 'blocked'}
{'step': 80, 'position': (2, 1), 'action': 'Left', 'outcome': 'done'}
{'step': 81, 'position': (2, 0), 'action': 'Up', 'outcome': 'done'}
{'step': 82, 'position': (1, 0), 'action': 'Up', 'outcome': 'done'}
{'step': 83, 'position': (0, 0), 'action': 'Right', 'outcome': 'done'}
{'step': 84, 'positio